# Building an Assistant Agent for Paper Abstract Drafting

In this tutorial, we'll walk through the process of building an Assistant using the `Assistant` class. We'll cover the following topics:

1. Setting up the environment
2. Creating custom tools
3. Initializing the Assistant
4. Interacting with the Assistant
5. Advanced usage and customization

## Setting Up the Environment

First, let's import the necessary modules and classes:

In [1]:
from slackagents import Assistant, FunctionTool, OpenAILLM, BaseLLMConfig
from IPython.display import Markdown


## Creating Custom Tools

Let's create a simple custom tool that our Assistant can use:
1. ArXiv Search Tool
2. Abstract Writer Tool

### ArXiv Search Tool

The ArXiv Search Tool will allow the Assistant to search for papers on the ArXiv database. We'll use the `arxiv` package to interact with the ArXiv API.

Install the `arxiv` package:

In [ ]:
!pip install arxiv

In [2]:
# Define a function to search Wikipedia
"""ArXiv Query Tool with OpenAPI Specification."""
import arxiv
from typing import List
def arxiv_search(
    query: str, 
    sort_by: str = "relevance", 
    max_results: int = 3
) -> List[str]:
    """A tool to query arxiv.org.
    :param query: The query to be passed to arXiv.
    :type query: string
    :param sort_by: Either 'relevance' (default) or 'recent'
    :type sort_by: string
    :param max_results: The maximum number of results to return
    :type max_results: number
    :return: A list of search results
    :rtype: array
    """
    sort = arxiv.SortCriterion.Relevance
    if sort_by == "recent":
        sort = arxiv.SortCriterion.SubmittedDate
    search = arxiv.Search(
        query, max_results=max_results, sort_by=sort
    )
    results = []
    for result in search.results():
        search_str = f"{result.pdf_url}: {result.title}\n{result.summary}"
        results.append(search_str)
    return results

We can now create a tool from this function:

In [ ]:
import json
arxiv_tool = FunctionTool.from_function(arxiv_search)
print(f"ArXiv Search Tool: {json.dumps(arxiv_tool.info, indent=4)}")

## Paper Abstract Writing Tool

The Paper Abstract Writing Tool will allow the Assistant to write an abstract for a given topic. We'll use another method `from_pydantic` to define the tool. Users can definitely use `from_function` method above as well.





In [ ]:
from pydantic import BaseModel, Field
class AbstractWriterTool(BaseModel):
    
    topic: str = Field(..., description="The topic of the abstract")
    context: str = Field(..., description="The context of the topic from other sources (e.g, academic papers, etc.)")
    
    @classmethod
    def execute(self, topic: str, context: str) -> str:
        """Calculate the sum of two numbers."""
        prompt = f"""
        Write an abstract for the following topic: {topic}
        Context: {context}
        Be creative and write a compelling abstract.
        """
        llm = OpenAILLM(BaseLLMConfig(model="gpt-4o"))
        messages = [{"role": "user", "content": prompt}]
        response = llm.chat_completion(messages)["content"]
        return response

abstract_writer_tool = FunctionTool.from_pydantic(
    AbstractWriterTool, 
    name="abstract_writer_tool", 
    description="A tool to write an academic paper abstract for a given topic"
)
print(json.dumps(abstract_writer_tool.info, indent=4))


## Initializing the Assistant

We can now initialize the Assistant with our custom tools. We can also specify the other parameters, such as:
- `name`: name of the agent
- `description`: description of the agent
- `system prompt`: system instruction of the assistant (default to `BASE_ASSISTANT_PROMPT`)
- `llm`: language model (default to OpenAI GPT-4o)
- `verbose`: whether to print the assistant's tool request and response messages to the console

In [5]:
assistant = Assistant(
    name="Paper Guru",
    desc="An assistant that can help brainstorm an abstract for a given topic",
    llm=OpenAILLM(BaseLLMConfig(model="gpt-4o")),
    tools=[arxiv_tool, abstract_writer_tool],
    system_prompt="You are an AI assistant that can help brainstorm an abstract for a given topic.",
    verbose=True # whether to print the assistant's tool request and response messages to the console
)

### One Turn Example

Then, we can interact with the assistant with verbose mode on:

In [ ]:
message = "Can you write an abstract for the topic of efficient AI for LLM training?"
response = assistant.chat(message)
display(Markdown(response))

### Interactive Demo

Let's turn off the verbose mode for the interactive demo. We use Gradio to create the interactive demo in notebook. To run it, please install Gradio first:

In [ ]:
!pip install gradio

Running the following cell will trigger hosting of a chatbot interface at http://127.0.0.1:7860 using Gradio and visualize it inline in the notebook.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import gradio as gr

assistant = Assistant(
    name="Paper Guru",
    desc="An assistant that can help brainstorm an abstract for a given topic",
    llm=OpenAILLM(BaseLLMConfig(model="gpt-4o")),
    tools=[arxiv_tool, abstract_writer_tool],
    system_prompt="You are an AI assistant that can help brainstorm an abstract for a given topic.",
    verbose=False
)

def respond(message, history):
    return assistant.chat(message)


gr.ChatInterface(
    respond, 
    theme="soft",
    title="Paper Guru",
    description="Ask Paper Guru any research-related questions",
    type="messages"
).launch(inline=True, share=False)